# SLAM Mapping And Livox Visualization

This notebook combines the control flow from `/home/unitree/slam_example/src/keyDemo.cpp` with the Livox/SLAM point-cloud visualization helpers in `/home/unitree/EF/ef_ws/g1`.

`keyDemo` actions mapped here: start mapping, save map, load/relocate, capture current pose as a waypoint, execute the waypoint list, pause/resume navigation, and stop SLAM. The visualization subscribes to SLAM mapping/relocation clouds first and falls back to raw Livox MID-360 points.

In [1]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional, Tuple

import numpy as np

ROOT = Path('/home/unitree/EF/ef_ws/g1')
MODULES = ROOT / 'modules'
SCRIPTS = MODULES / 'scripts'
for path in (MODULES, SCRIPTS):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelSubscriber
from unitree_sdk2py.idl.sensor_msgs.msg.dds_ import PointCloud2_
from sdk_slam import SlamInfoSubscriber, SlamOdomSubscriber, SlamOperateClient

IFACE = 'eth0'
DOMAIN_ID = 0
MAP_PATH = '/home/unitree/test.pcd'

SLAM_POINTS_TOPIC = 'rt/unitree/slam_mapping/points'
SLAM_RELOCATION_POINTS_TOPIC = 'rt/unitree/slam_relocation/points'
SLAM_GLOBAL_MAP_TOPIC = 'rt/unitree/slam_relocation/global_map'
LIVOX_POINTS_TOPIC = 'rt/utlidar/cloud_livox_mid360'

ChannelFactoryInitialize(DOMAIN_ID, IFACE)
print(f'DDS initialized on iface={IFACE!r}, domain={DOMAIN_ID}')

DDS initialized on iface='eth0', domain=0


## SLAM Controller

This is a Python notebook version of the C++ `TestClient`: it uses the same `slam_operate` API IDs through `SlamOperateClient`, reads `rt/slam_info` and `rt/slam_key_info`, stores captured poses in memory, and sends those poses to `pose_nav`.

In [2]:
@dataclass
class PoseTarget:
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0
    q_x: float = 0.0
    q_y: float = 0.0
    q_z: float = 0.0
    q_w: float = 1.0
    mode: int = 1

    @classmethod
    def from_xy_yaw(cls, x: float, y: float, yaw: float, z: float = 0.0, mode: int = 1) -> 'PoseTarget':
        return cls(
            x=float(x),
            y=float(y),
            z=float(z),
            q_z=math.sin(float(yaw) * 0.5),
            q_w=math.cos(float(yaw) * 0.5),
            mode=int(mode),
        )

    def yaw(self) -> float:
        siny_cosp = 2.0 * (self.q_w * self.q_z + self.q_x * self.q_y)
        cosy_cosp = 1.0 - 2.0 * (self.q_y * self.q_y + self.q_z * self.q_z)
        return math.atan2(siny_cosp, cosy_cosp)

    def as_nav_payload(self) -> dict[str, Any]:
        return {
            'x': self.x,
            'y': self.y,
            'z': self.z,
            'q_x': self.q_x,
            'q_y': self.q_y,
            'q_z': self.q_z,
            'q_w': self.q_w,
        }


def _parse_current_pose(payload_raw: Optional[str]) -> Optional[PoseTarget]:
    if not payload_raw:
        return None
    try:
        payload = json.loads(payload_raw)
        if int(payload.get('errorCode', 0)) != 0:
            return None
        cur = payload.get('data', {}).get('currentPose', {})
        x = float(cur.get('x', 0.0))
        y = float(cur.get('y', 0.0))
        z = float(cur.get('z', 0.0))
        if {'q_x', 'q_y', 'q_z', 'q_w'} <= set(cur):
            return PoseTarget(x, y, z, float(cur['q_x']), float(cur['q_y']), float(cur['q_z']), float(cur['q_w']))
        yaw = float(cur.get('yaw', 0.0))
        return PoseTarget.from_xy_yaw(x, y, yaw, z=z)
    except Exception:
        return None


def _response_dict(resp: Any) -> dict[str, Any]:
    raw = resp.raw
    try:
        raw = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        pass
    ok = int(resp.code) == 0
    if isinstance(raw, dict):
        ok = ok and int(raw.get('errorCode', 0)) == 0 and bool(raw.get('succeed', True))
    return {'code': int(resp.code), 'ok': bool(ok), 'raw': raw}


def _print_slam_response(label: str, result: dict[str, Any]) -> dict[str, Any]:
    print(f'[{label}]')
    print(f"statusCode: {result['code']}")
    print(f"ok: {result['ok']}")
    raw = result.get('raw')
    print('data:', json.dumps(raw, indent=2, sort_keys=True) if isinstance(raw, dict) else raw)
    if isinstance(raw, dict) and int(raw.get('errorCode', 0)) != 0:
        print(f"SLAM error: errorCode={raw.get('errorCode')} info={raw.get('info', '')}")
    elif not result.get('ok'):
        print('SLAM operation did not report success.')
    return result


def _parse_slam_info_error(payload_raw: Optional[str]) -> Optional[str]:
    if not payload_raw:
        return None
    try:
        payload = json.loads(payload_raw)
    except Exception as exc:
        return f'Invalid SLAM JSON: {exc}'
    if isinstance(payload, dict) and int(payload.get('errorCode', 0)) != 0:
        return f"SLAM topic errorCode={payload.get('errorCode')} info={payload.get('info', '')}"
    return None


class SlamNotebookController:
    def __init__(self, map_path: str = MAP_PATH, timeout_s: float = 10.0) -> None:
        self.map_path = str(map_path)
        self.client = SlamOperateClient()
        self.client.Init()
        self.client.SetTimeout(float(timeout_s))
        self.info = SlamInfoSubscriber('rt/slam_info', 'rt/slam_key_info')
        self.info.start()
        self.odom = SlamOdomSubscriber()
        self.odom.start()
        self.pose_list: list[PoseTarget] = []
        self.initial_pose: Optional[PoseTarget] = None
        self.relocation_ready = False

    def current_pose(self) -> Optional[PoseTarget]:
        return _parse_current_pose(self.info.get_info()) or _parse_current_pose(self.info.get_key())

    def start_mapping(self, slam_type: str = 'indoor') -> dict[str, Any]:
        self.pose_list.clear()
        self.relocation_ready = False
        self.capture_initial_pose()
        return _print_slam_response('start_mapping (1801)', _response_dict(self.client.start_mapping(slam_type=slam_type)))

    def save_map(self, map_path: Optional[str] = None) -> dict[str, Any]:
        if map_path is not None:
            self.map_path = str(map_path)
        return _print_slam_response('end_mapping (1802)', _response_dict(self.client.end_mapping(self.map_path)))

    def relocate(self, map_path: Optional[str] = None, pose: Optional[PoseTarget] = None) -> dict[str, Any]:
        if map_path is not None:
            self.map_path = str(map_path)
        pose = pose or self.current_pose() or PoseTarget()
        resp = self.client.init_pose(pose.x, pose.y, pose.z, pose.q_x, pose.q_y, pose.q_z, pose.q_w, self.map_path)
        result = _response_dict(resp)
        self.relocation_ready = bool(result['ok'])
        if self.relocation_ready:
            self.initial_pose = pose
        if not self.relocation_ready:
            print('Relocation did not start. Do not add or execute navigation poses yet.')
        return _print_slam_response('init_pose / relocation (1804)', result)

    def capture_initial_pose(self) -> Optional[PoseTarget]:
        pose = self.current_pose()
        if pose is None:
            print('Initial pose: <none>')
            return None
        self.initial_pose = pose
        print(f'Initial pose: x={pose.x:.3f} y={pose.y:.3f} z={pose.z:.3f} yaw={pose.yaw():.3f}')
        return pose

    def add_current_pose(self) -> PoseTarget:
        if not self.relocation_ready:
            raise RuntimeError('Start relocation successfully before adding navigation poses.')
        pose = self.current_pose()
        if pose is None:
            raise RuntimeError('No valid current pose received from rt/slam_info yet.')
        self.pose_list.append(pose)
        print(f'Added pose: x={pose.x:.3f} y={pose.y:.3f} z={pose.z:.3f} yaw={pose.yaw():.3f}')
        return pose

    def add_xy_yaw(self, x: float, y: float, yaw: float, mode: int = 1) -> PoseTarget:
        pose = PoseTarget.from_xy_yaw(x, y, yaw, mode=mode)
        self.pose_list.append(pose)
        return pose

    def clear_task_list(self) -> None:
        self.pose_list.clear()
        print('Clear task list')

    def navigate_to(self, pose: PoseTarget) -> dict[str, Any]:
        resp = self.client.pose_nav(pose.x, pose.y, pose.z, pose.q_x, pose.q_y, pose.q_z, pose.q_w, mode=pose.mode)
        return _print_slam_response('pose_nav (1102)', _response_dict(resp))

    def execute_task_list(self, wait_for_result: bool = True, timeout_s: float = 60.0) -> list[dict[str, Any]]:
        if not self.pose_list:
            raise RuntimeError('No navigation poses queued.')
        if not self.relocation_ready:
            raise RuntimeError('Relocation is not active.')
        results = []
        for idx, pose in enumerate(self.pose_list, start=1):
            result = self.navigate_to(pose)
            result['target_index'] = idx
            result['target'] = pose.as_nav_payload()
            if wait_for_result:
                result['task_result'] = self.wait_for_task_result(timeout_s=timeout_s)
                task = result['task_result']
                if task is None:
                    print('task_result: timeout waiting for rt/slam_key_info')
                elif task.get('data', {}).get('is_arrived'):
                    print(f"I arrived {task.get('data', {}).get('targetNodeName')}")
                else:
                    print(f"I not arrived {task.get('data', {}).get('targetNodeName')}  Please help me!!")
            results.append(result)
            if result['code'] != 0:
                break
        return results

    def wait_for_task_result(self, timeout_s: float = 60.0) -> Optional[dict[str, Any]]:
        deadline = time.time() + float(timeout_s)
        last_key = None
        while time.time() < deadline:
            key = self.info.get_key()
            if key and key != last_key:
                last_key = key
                try:
                    payload = json.loads(key)
                    if payload.get('type') == 'task_result':
                        return payload
                except Exception:
                    pass
            time.sleep(0.05)
        return None

    def pause_nav(self) -> dict[str, Any]:
        return _print_slam_response('pause_nav (1201)', _response_dict(self.client.pause_nav()))

    def resume_nav(self) -> dict[str, Any]:
        return _print_slam_response('resume_nav (1202)', _response_dict(self.client.resume_nav()))

    def stop_slam(self) -> dict[str, Any]:
        self.relocation_ready = False
        return _print_slam_response('close_slam (1901)', _response_dict(self.client.close_slam()))

    def status(self) -> dict[str, Any]:
        pose = self.current_pose()
        odom_pose = self.odom.get_pose()
        info_error = _parse_slam_info_error(self.info.get_info())
        key_error = _parse_slam_info_error(self.info.get_key())
        if info_error:
            print(info_error)
        if key_error:
            print(key_error)
        if pose is None:
            print('SLAM pose: <none>')
        else:
            print(f'SLAM pose: x={pose.x:.3f} y={pose.y:.3f} z={pose.z:.3f} yaw={pose.yaw():.3f}')
        return {
            'map_path': self.map_path,
            'relocation_ready': self.relocation_ready,
            'initial_pose': None if self.initial_pose is None else {'x': self.initial_pose.x, 'y': self.initial_pose.y, 'z': self.initial_pose.z, 'yaw': self.initial_pose.yaw()},
            'current_pose': None if pose is None else {'x': pose.x, 'y': pose.y, 'z': pose.z, 'yaw': pose.yaw()},
            'odom_pose': odom_pose,
            'queued_targets': [p.as_nav_payload() for p in self.pose_list],
            'slam_info': self.info.get_info(),
            'slam_key_info': self.info.get_key(),
            'slam_info_error': info_error,
            'slam_key_info_error': key_error,
        }


slam = SlamNotebookController(MAP_PATH)
slam.status()

SLAM pose: <none>


{'map_path': '/home/unitree/test.pcd',
 'relocation_ready': False,
 'initial_pose': None,
 'current_pose': None,
 'odom_pose': None,
 'queued_targets': [],
 'slam_info': None,
 'slam_key_info': None,
 'slam_info_error': None,
 'slam_key_info_error': None}

## Mapping Workflow

Run these cells in order for the same basic flow as the C++ demo: start mapping (`q`), save map (`w`), relocate from the saved map (`a`), add poses (`s`), execute (`d`), pause/resume (`z`/`x`), and stop.

In [20]:
# q in keyDemo: start mapping.
slam.start_mapping('indoor')

Initial pose: x=0.128 y=0.074 z=0.048 yaw=-0.158
[start_mapping (1801)]
statusCode: 0
ok: True
data: {
  "data": {},
  "errorCode": 0,
  "info": "Successfully started mapping.",
  "succeed": true
}


{'code': 0,
 'ok': True,
 'raw': {'succeed': True,
  'errorCode': 0,
  'info': 'Successfully started mapping.',
  'data': {}}}

In [21]:
# w in keyDemo: end mapping and save the map.
slam.save_map('/home/unitree/test.pcd')

[end_mapping (1802)]
statusCode: 0
ok: True
data: {
  "data": {},
  "errorCode": 0,
  "info": "Save pcd successfully.",
  "succeed": true
}


{'code': 0,
 'ok': True,
 'raw': {'succeed': True,
  'errorCode': 0,
  'info': 'Save pcd successfully.',
  'data': {}}}

In [22]:
# a in keyDemo: start relocation from a saved map.
slam.relocate('/home/unitree/test.pcd')

[init_pose / relocation (1804)]
statusCode: 0
ok: True
data: {
  "data": {},
  "errorCode": 0,
  "info": "Successfully started re-location.",
  "succeed": true
}


{'code': 0,
 'ok': True,
 'raw': {'succeed': True,
  'errorCode': 0,
  'info': 'Successfully started re-location.',
  'data': {}}}

In [24]:
# s in keyDemo: add the current SLAM pose to the task list.
# Move the robot to each desired waypoint, then run this once per waypoint.
slam.add_current_pose()

# Alternative: enter explicit map-frame targets.
# slam.add_xy_yaw(1.0, 0.0, 0.0)
# slam.add_xy_yaw(1.0, 0.5, math.radians(90))

slam.pose_list

Added pose: x=-0.029 y=-0.012 z=0.042 yaw=-0.325


[PoseTarget(x=0.25267791534102174, y=-0.1456076032199863, z=0.035782816395148076, q_x=0.015078731892583143, q_y=-0.05935297079733407, q_z=0.11413571223273573, q_w=-0.9915759657712406, mode=1),
 PoseTarget(x=-0.02917011783190503, y=-0.012468302377543499, z=0.04226231159649283, q_x=-0.012278785630962715, q_y=0.07945407456309365, q_z=-0.1592085037695439, q_w=0.9839659210492071, mode=1)]

In [27]:
# d in keyDemo: execute queued poses. This waits for task_result messages from rt/slam_key_info.
slam.execute_task_list(wait_for_result=True, timeout_s=60.0)

[pose_nav (1102)]
statusCode: 3104
ok: False
data: None
SLAM operation did not report success.
I arrived 9999


[{'code': 3104,
  'ok': False,
  'raw': None,
  'target_index': 1,
  'target': {'x': 0.25267791534102174,
   'y': -0.1456076032199863,
   'z': 0.035782816395148076,
   'q_x': 0.015078731892583143,
   'q_y': -0.05935297079733407,
   'q_z': 0.11413571223273573,
   'q_w': -0.9915759657712406},
  'task_result': {'type': 'task_result',
   'sec': 1779870379,
   'nanosec': 230769334,
   'errorCode': 0,
   'info': '',
   'data': {'targetNodeName': 9999, 'is_arrived': True}}}]

In [ ]:
# z / x / f / other in keyDemo.
# slam.pause_nav()
# slam.resume_nav()
# slam.clear_task_list()
# slam.stop_slam()

## Point-Cloud Subscribers

The viewer uses the same topic preference as `modules/scripts/slam_points_viewer.py`: SLAM mapping points first, relocation/global-map points next, and raw Livox MID-360 points as the fallback.

In [8]:
class PointCloudSubscriber:
    def __init__(self, topic: str) -> None:
        self.topic = str(topic)
        self.msg: Optional[PointCloud2_] = None
        self.ts = 0.0
        self.sub: Optional[ChannelSubscriber] = None

    def start(self) -> None:
        if self.sub is None:
            self.sub = ChannelSubscriber(self.topic, PointCloud2_)
            self.sub.Init(self._callback, 10)

    def _callback(self, msg: PointCloud2_) -> None:
        self.msg = msg
        self.ts = time.time()

    def latest(self) -> Tuple[Optional[PointCloud2_], float]:
        return self.msg, self.ts


def decode_points_xyz(msg: PointCloud2_, stride: int = 4, zmin: float = -1.0, zmax: float = 2.5, max_points: int = 60000) -> np.ndarray:
    fields = {field.name: field for field in msg.fields}
    if not {'x', 'y', 'z'} <= set(fields):
        print(f'PointCloud2 decode failed: missing xyz fields; fields={list(fields)}')
        return np.empty((0, 3), dtype=np.float64)
    point_step = int(msg.point_step)
    if point_step <= 0:
        print(f'PointCloud2 decode failed: invalid point_step={point_step}')
        return np.empty((0, 3), dtype=np.float64)
    data = bytes(msg.data)
    if not data:
        print('PointCloud2 decode failed: null/empty data buffer.')
        return np.empty((0, 3), dtype=np.float64)
    dtype = np.dtype({
        'names': ['x', 'y', 'z'],
        'formats': ['<f4', '<f4', '<f4'],
        'offsets': [int(fields['x'].offset), int(fields['y'].offset), int(fields['z'].offset)],
        'itemsize': point_step,
    })
    arr = np.frombuffer(data, dtype=dtype, count=len(data) // point_step)
    step = max(1, int(stride))
    pts = np.stack([arr['x'][::step], arr['y'][::step], arr['z'][::step]], axis=1).astype(np.float64)
    mask = np.isfinite(pts).all(axis=1) & (pts[:, 2] >= float(zmin)) & (pts[:, 2] <= float(zmax))
    pts = pts[mask]
    if pts.size == 0:
        print(f'PointCloud2 decode warning: zero usable xyz points after stride/filter zmin={zmin} zmax={zmax}.')
    if max_points and pts.shape[0] > int(max_points):
        idx = np.linspace(0, pts.shape[0] - 1, int(max_points), dtype=np.int64)
        pts = pts[idx]
    return pts


cloud_subs = [
    PointCloudSubscriber(SLAM_POINTS_TOPIC),
    PointCloudSubscriber(SLAM_RELOCATION_POINTS_TOPIC),
    PointCloudSubscriber(SLAM_GLOBAL_MAP_TOPIC),
    PointCloudSubscriber(LIVOX_POINTS_TOPIC),
]
for sub in cloud_subs:
    sub.start()

def latest_cloud() -> Tuple[Optional[str], Optional[PointCloud2_], float]:
    live = [(sub.topic, *sub.latest()) for sub in cloud_subs if sub.latest()[0] is not None]
    if not live:
        print('No point cloud messages received from SLAM mapping, relocation, global map, or Livox topics.')
        return None, None, 0.0
    return max(live, key=lambda item: item[2])


def cloud_status() -> list[dict[str, Any]]:
    now = time.time()
    rows = []
    for sub in cloud_subs:
        msg, ts = sub.latest()
        width = None if msg is None else int(getattr(msg, 'width', 0) or 0)
        height = None if msg is None else int(getattr(msg, 'height', 0) or 0)
        point_step = None if msg is None else int(getattr(msg, 'point_step', 0) or 0)
        data_len = None if msg is None else len(bytes(getattr(msg, 'data', b'')))
        ok = msg is not None and width not in (None, 0) and data_len not in (None, 0) and point_step not in (None, 0)
        rows.append({
            'topic': sub.topic,
            'age_s': None if ts <= 0 else round(now - ts, 3),
            'ok': ok,
            'width': width,
            'height': height,
            'point_step': point_step,
            'data_bytes': data_len,
            'frame_id': None if msg is None else getattr(getattr(msg, 'header', None), 'frame_id', None),
        })
    for row in rows:
        if row['ok']:
            print(f"[cloud ok] {row['topic']} width={row['width']} data_bytes={row['data_bytes']} age_s={row['age_s']}")
        elif row['width'] == 0 or row['data_bytes'] == 0:
            print(f"[cloud empty] {row['topic']} width={row['width']} data_bytes={row['data_bytes']} frame_id={row['frame_id']}")
        else:
            print(f"[cloud none] {row['topic']} no message received")
    return rows


time.sleep(1.0)
cloud_status()

[cloud none] rt/unitree/slam_mapping/points no message received
[cloud ok] rt/unitree/slam_relocation/points width=812 data_bytes=38976 age_s=0.019
[cloud none] rt/unitree/slam_relocation/global_map no message received
[cloud ok] rt/utlidar/cloud_livox_mid360 width=20064 data_bytes=441408 age_s=0.035


[{'topic': 'rt/unitree/slam_mapping/points',
  'age_s': None,
  'ok': False,
  'width': None,
  'height': None,
  'point_step': None,
  'data_bytes': None,
  'frame_id': None},
 {'topic': 'rt/unitree/slam_relocation/points',
  'age_s': 0.019,
  'ok': True,
  'width': 812,
  'height': 1,
  'point_step': 48,
  'data_bytes': 38976,
  'frame_id': 'map'},
 {'topic': 'rt/unitree/slam_relocation/global_map',
  'age_s': None,
  'ok': False,
  'width': None,
  'height': None,
  'point_step': None,
  'data_bytes': None,
  'frame_id': None},
 {'topic': 'rt/utlidar/cloud_livox_mid360',
  'age_s': 0.035,
  'ok': True,
  'width': 20064,
  'height': 1,
  'point_step': 22,
  'data_bytes': 441408,
  'frame_id': 'livox_frame'}]

## Static Notebook View

This renders the newest cloud frame in Plotly. During active mapping it should prefer `rt/unitree/slam_mapping/points`; outside mapping it may show the raw Livox topic.

In [9]:
import plotly.graph_objects as go


def show_latest_cloud(stride: int = 4, max_points: int = 50000, zmin: float = -1.0, zmax: float = 2.5) -> Optional[go.Figure]:
    topic, msg, ts = latest_cloud()
    if msg is None:
        print('No SLAM or Livox point cloud received yet.')
        return None
    pts = decode_points_xyz(msg, stride=stride, zmin=zmin, zmax=zmax, max_points=max_points)
    if pts.size == 0:
        print(f'Latest cloud on {topic} decoded to zero usable points.')
        return None
    pose = slam.current_pose()
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', name=topic,
        marker={'size': 1.6, 'color': pts[:, 2], 'colorscale': 'Viridis', 'opacity': 0.85, 'colorbar': {'title': 'z (m)'}},
        hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>',
    ))
    if pose is not None:
        fig.add_trace(go.Scatter3d(
            x=[pose.x], y=[pose.y], z=[pose.z], mode='markers', name='SLAM pose',
            marker={'size': 7, 'color': 'red', 'symbol': 'diamond'},
        ))
    if slam.initial_pose is not None:
        fig.add_trace(go.Scatter3d(
            x=[slam.initial_pose.x], y=[slam.initial_pose.y], z=[slam.initial_pose.z], mode='markers', name='initial pose',
            marker={'size': 9, 'color': 'orange', 'symbol': 'circle'},
        ))
    if slam.pose_list:
        fig.add_trace(go.Scatter3d(
            x=[p.x for p in slam.pose_list], y=[p.y for p in slam.pose_list], z=[p.z for p in slam.pose_list],
            mode='markers+lines', name='queued targets',
            marker={'size': 5, 'color': 'black'}, line={'color': 'black', 'width': 4},
        ))
    fig.update_layout(
        title=f'{topic} - {len(pts)} decoded points - age {time.time() - ts:.2f}s',
        scene={'xaxis_title': 'x (m)', 'yaxis_title': 'y (m)', 'zaxis_title': 'z (m)', 'aspectmode': 'data'},
        width=950, height=750, margin={'l': 0, 'r': 0, 't': 40, 'b': 0},
    )
    fig.show()
    return fig


# show_latest_cloud()

## Live Plotly View

This updates a `FigureWidget` in-place. It is lighter than repeatedly calling `fig.show()`, but still keep `max_points` modest on the robot.

In [11]:
from IPython.display import display


def _task_xyz() -> Tuple[list, list, list]:
    return ([p.x for p in slam.pose_list], [p.y for p in slam.pose_list], [p.z for p in slam.pose_list])


def _pose_xyz(pose: Optional[PoseTarget]) -> Tuple[list, list, list]:
    if pose is None:
        return ([], [], [])
    return ([pose.x], [pose.y], [pose.z])


def _clicked_xyz(trace: Any, points: Any) -> Optional[Tuple[float, float, float]]:
    if getattr(points, 'xs', None) and getattr(points, 'ys', None):
        z_values = getattr(points, 'zs', None) or [0.0]
        return (float(points.xs[0]), float(points.ys[0]), float(z_values[0]))
    inds = getattr(points, 'point_inds', None) or []
    if not inds:
        return None
    idx = int(inds[0])
    try:
        return (float(trace.x[idx]), float(trace.y[idx]), float(trace.z[idx]))
    except Exception:
        return None


def _refresh_pose_and_task_traces(fig: go.FigureWidget) -> None:
    current_pose = slam.current_pose()
    cx, cy, cz = _pose_xyz(current_pose)
    ix, iy, iz = _pose_xyz(slam.initial_pose)
    tx, ty, tz = _task_xyz()
    with fig.batch_update():
        fig.data[1].x = cx
        fig.data[1].y = cy
        fig.data[1].z = cz
        fig.data[2].x = ix
        fig.data[2].y = iy
        fig.data[2].z = iz
        fig.data[3].x = tx
        fig.data[3].y = ty
        fig.data[3].z = tz


def clickable_task_map(stride: int = 6, max_points: int = 25000, zmin: float = -1.0, zmax: float = 2.5) -> go.FigureWidget:
    topic, msg, ts = latest_cloud()
    pts = np.empty((0, 3), dtype=float) if msg is None else decode_points_xyz(msg, stride=stride, zmin=zmin, zmax=zmax, max_points=max_points)
    if pts.size == 0:
        pts = np.zeros((1, 3), dtype=float)
        print('Clickable map has no real cloud points yet; run cloud_status() and retry after SLAM/Livox publishes data.')

    cx, cy, cz = _pose_xyz(slam.current_pose())
    ix, iy, iz = _pose_xyz(slam.initial_pose)
    tx, ty, tz = _task_xyz()

    fig = go.FigureWidget(data=[
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', name='clickable cloud',
            marker={'size': 1.8, 'color': pts[:, 2], 'colorscale': 'Viridis', 'opacity': 0.82},
            hovertemplate='click to add task<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>',
        ),
        go.Scatter3d(
            x=cx, y=cy, z=cz, mode='markers', name='current SLAM pose',
            marker={'size': 8, 'color': 'red', 'symbol': 'diamond'},
        ),
        go.Scatter3d(
            x=ix, y=iy, z=iz, mode='markers', name='initial pose',
            marker={'size': 10, 'color': 'orange', 'symbol': 'circle'},
        ),
        go.Scatter3d(
            x=tx, y=ty, z=tz, mode='markers+lines', name='task points',
            marker={'size': 7, 'color': 'black', 'symbol': 'circle'}, line={'color': 'black', 'width': 5},
        ),
    ])
    fig.update_layout(
        scene={'xaxis_title': 'x (m)', 'yaxis_title': 'y (m)', 'zaxis_title': 'z (m)', 'aspectmode': 'data'},
        width=950, height=750, margin={'l': 0, 'r': 0, 't': 45, 'b': 0},
        title=f'Click a cloud point to add a task point - {topic or "no cloud"}',
    )

    def on_cloud_click(trace, points, selector):
        clicked = _clicked_xyz(trace, points)
        if clicked is None:
            print('Click did not contain a map coordinate.')
            return
        x, y, z = clicked
        current_pose = slam.current_pose()
        yaw = 0.0 if current_pose is None else current_pose.yaw()
        pose = slam.add_xy_yaw(x, y, yaw)
        pose.z = z
        print(f'Clicked task point added: x={pose.x:.3f} y={pose.y:.3f} z={pose.z:.3f} yaw={pose.yaw():.3f}')
        _refresh_pose_and_task_traces(fig)

    fig.data[0].on_click(on_cloud_click)
    display(fig)
    return fig


def refresh_clickable_task_map(fig: go.FigureWidget, stride: int = 6, max_points: int = 25000, zmin: float = -1.0, zmax: float = 2.5) -> go.FigureWidget:
    topic, msg, ts = latest_cloud()
    if msg is not None:
        pts = decode_points_xyz(msg, stride=stride, zmin=zmin, zmax=zmax, max_points=max_points)
        if pts.size:
            with fig.batch_update():
                fig.data[0].x = pts[:, 0]
                fig.data[0].y = pts[:, 1]
                fig.data[0].z = pts[:, 2]
                fig.data[0].marker.color = pts[:, 2]
                fig.layout.title.text = f'Click a cloud point to add a task point - {topic} - pts={len(pts)}'
    _refresh_pose_and_task_traces(fig)
    return fig


def live_cloud_view(duration_s: float = 60.0, refresh_s: float = 0.5, stride: int = 6, max_points: int = 20000):
    fig = clickable_task_map(stride=stride, max_points=max_points)
    t0 = time.time()
    while time.time() - t0 < float(duration_s):
        refresh_clickable_task_map(fig, stride=stride, max_points=max_points)
        time.sleep(max(0.05, float(refresh_s)))
    return fig


task_fig = clickable_task_map()
refresh_clickable_task_map(task_fig)
live_cloud_view(duration_s=60, refresh_s=0.5)


FigureWidget({
    'data': [{'hovertemplate': 'click to add task<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>',
              'marker': {'color': {'bdata': ('AAAAoA7V7L8AAACgLkHpvwAAAAAxru' ... 'BGrOq/AAAAgCtf5r8AAACg/SPvvw=='),
                                   'dtype': 'f8'},
                         'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                        '#482878'], [0.2222222222222222,
                                        '#3e4989'], [0.3333333333333333,
                                        '#31688e'], [0.4444444444444444,
                                        '#26828e'], [0.5555555555555556,
                                        '#1f9e89'], [0.6666666666666666,
                                        '#35b779'], [0.7777777777777778,
                                        '#6ece58'], [0.8888888888888888,
                                        '#b5de2b'], [1.0, '#fde725']],
                         'opacity': 0.82,
     

FigureWidget({
    'data': [{'hovertemplate': 'click to add task<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>',
              'marker': {'color': {'bdata': ('AAAAAHQJ778AAACAjj/uvwAAAEC+7u' ... 'AAAFLu7r8AAABAqLLnvwAAACBQA++/'),
                                   'dtype': 'f8'},
                         'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                        '#482878'], [0.2222222222222222,
                                        '#3e4989'], [0.3333333333333333,
                                        '#31688e'], [0.4444444444444444,
                                        '#26828e'], [0.5555555555555556,
                                        '#1f9e89'], [0.6666666666666666,
                                        '#35b779'], [0.7777777777777778,
                                        '#6ece58'], [0.8888888888888888,
                                        '#b5de2b'], [1.0, '#fde725']],
                         'opacity': 0.82,
     

FigureWidget({
    'data': [{'hovertemplate': 'click to add task<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>',
              'marker': {'color': {'bdata': ('AAAAYJZu4r8AAACg33DYvwAAAOApi+' ... 'CzieC/AAAAAHFh4L8AAAAgiv3Vvw=='),
                                   'dtype': 'f8'},
                         'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                        '#482878'], [0.2222222222222222,
                                        '#3e4989'], [0.3333333333333333,
                                        '#31688e'], [0.4444444444444444,
                                        '#26828e'], [0.5555555555555556,
                                        '#1f9e89'], [0.6666666666666666,
                                        '#35b779'], [0.7777777777777778,
                                        '#6ece58'], [0.8888888888888888,
                                        '#b5de2b'], [1.0, '#fde725']],
                         'opacity': 0.82,
     

## Optional Open3D Viewer

`modules/scripts/slam_points_viewer.py` is still the best full-window viewer in this workspace. Run this from the notebook only when a desktop/OpenGL session is available.

In [12]:
viewer_proc = None


def launch_open3d_viewer() -> subprocess.Popen:
    global viewer_proc
    cmd = [
        sys.executable,
        str(SCRIPTS / 'slam_points_viewer.py'),
        '--iface', IFACE,
        '--domain-id', str(DOMAIN_ID),
        '--slam-points-topic', SLAM_POINTS_TOPIC,
        '--lidar-points-topic', LIVOX_POINTS_TOPIC,
    ]
    viewer_proc = subprocess.Popen(cmd, cwd=str(ROOT))
    return viewer_proc


viewer_proc = launch_open3d_viewer()
viewer_proc.terminate()

## Quick Debug Cells

In [13]:
slam.status()

SLAM pose: x=0.124 y=0.073 z=0.047 yaw=-0.157


{'map_path': '/home/unitree/test.pcd',
 'relocation_ready': True,
 'initial_pose': {'x': 0.0, 'y': 0.0, 'z': 0.0, 'yaw': 0.0},
 'current_pose': {'x': 0.12400958478275433,
  'y': 0.07293125003653056,
  'z': 0.04667096553097088,
  'yaw': -0.15662456608106068},
 'odom_pose': None,
 'queued_targets': [{'x': 0.10312135529667782,
   'y': 0.03444144897879673,
   'z': 0.053838505788430566,
   'q_x': -0.006230014157119642,
   'q_y': 0.06330045774245226,
   'q_z': -0.06381544391931343,
   'q_w': 0.9959326423461474}],
 'slam_info': '{"type":"pos_info","sec":1779870168,"nanosec":535611136,"errorCode":0,"info":"","data":{"currentPose":{"x":0.12400958478275433,"y":0.07293125003653056,"z":0.04667096553097088,"q_w":0.9954994617649683,"q_x":-0.0038699054396643456,"q_y":0.05414020498091568,"q_z":-0.07768322638878493},"pcdName":"test","address":"/home/unitree/test.pcd"}}',
 'slam_key_info': '{"type":"task_result","sec":1779870086,"nanosec":653176384,"errorCode":0,"info":"","data":{"targetNodeName":9999,"

In [14]:
cloud_status()

[cloud none] rt/unitree/slam_mapping/points no message received
[cloud ok] rt/unitree/slam_relocation/points width=823 data_bytes=39504 age_s=0.003
[cloud none] rt/unitree/slam_relocation/global_map no message received
[cloud ok] rt/utlidar/cloud_livox_mid360 width=20064 data_bytes=441408 age_s=0.123


[{'topic': 'rt/unitree/slam_mapping/points',
  'age_s': None,
  'ok': False,
  'width': None,
  'height': None,
  'point_step': None,
  'data_bytes': None,
  'frame_id': None},
 {'topic': 'rt/unitree/slam_relocation/points',
  'age_s': 0.003,
  'ok': True,
  'width': 823,
  'height': 1,
  'point_step': 48,
  'data_bytes': 39504,
  'frame_id': 'map'},
 {'topic': 'rt/unitree/slam_relocation/global_map',
  'age_s': None,
  'ok': False,
  'width': None,
  'height': None,
  'point_step': None,
  'data_bytes': None,
  'frame_id': None},
 {'topic': 'rt/utlidar/cloud_livox_mid360',
  'age_s': 0.123,
  'ok': True,
  'width': 20064,
  'height': 1,
  'point_step': 22,
  'data_bytes': 441408,
  'frame_id': 'livox_frame'}]

In [15]:
from modules.sdk_client import Robot

In [16]:
robot = Robot("eth0")

In [19]:
robot.rgbd_host = "192.168.2.41"
robot.get_rgbd()

{'source': 'zmq://192.168.2.41:5555',
 'topic': '',
 'timestamp': 1779870499.704521,
 'rgb_jpeg': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x01\x01\x01\x01\x02\x01\x01\x01\x02\x02\x02\x02\x02\x04\x03\x02\x02\x02\x02\x05\x04\x04\x03\x04\x06\x05\x06\x06\x06\x05\x06\x06\x06\x07\t\x08\x06\x07\t\x07\x06\x06\x08\x0b\x08\t\n\n\n\n\n\x06\x08\x0b\x0c\x0b\n\x0c\t\n\n\n\xff\xdb\x00C\x01\x02\x02\x02\x02\x02\x02\x05\x03\x03\x05\n\x07\x06\x07\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\xff\xc0\x00\x11\x08\x01\xe0\x02\x80\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'(